# **Parameters**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType, StructType, ArrayType

spark = SparkSession.builder.getOrCreate()


nome_tabella_bronze = "bronze_product_catalog"   # es. @item().DestinationTable
source_folder       = "product_catalog"          # es. @item().SourceFolder

BRONZE_LAKEHOUSE = "LH_Bronze"
SILVER_LAKEHOUSE = "LH_Silver"

nome_tabella_silver = "silver_product_catalog"

print(f"📥 Tabella sorgente (Bronze) : {nome_tabella_bronze}")
print(f"📤 Tabella destinazione (Silver): {nome_tabella_silver}")
print(f"🗂  Source folder: {source_folder}")


# **Path ABFSS**

In [ ]:
bronze_info = notebookutils.lakehouse.get(BRONZE_LAKEHOUSE)
bronze_path = bronze_info["properties"]["abfsPath"]

silver_info = notebookutils.lakehouse.get(SILVER_LAKEHOUSE)
silver_path = silver_info["properties"]["abfsPath"]

print(f"📂 Bronze path : {bronze_path}")
print(f"📂 Silver path : {silver_path}")


# **Recursive Cleaning Function**

In [ ]:

def cleaning_expression(campo_dataType, espressione_colonna):
    if isinstance(campo_dataType, StringType):
        col_pulita = F.trim(espressione_colonna)
        return F.when(col_pulita == "", None).otherwise(col_pulita)

    elif isinstance(campo_dataType, StructType):
        campi_puliti = [
            cleaning_expression(
                sotto_campo.dataType,
                espressione_colonna.getField(sotto_campo.name)
            ).alias(sotto_campo.name)
            for sotto_campo in campo_dataType.fields
        ]
        return F.struct(*campi_puliti)

    elif isinstance(campo_dataType, ArrayType):
        tipo_elemento = campo_dataType.elementType
        return F.transform(
            espressione_colonna,
            lambda x: cleaning_expression(tipo_elemento, x)
        )

    else:
        return espressione_colonna

def cleaning_dataframe(df: DataFrame) -> DataFrame:
    espressioni = [
        cleaning_expression(campo.dataType, F.col(campo.name)).alias(campo.name)
        for campo in df.schema.fields
    ]
    df_pulito = df.select(*espressioni)
    df_pulito = df_pulito.dropna(how="all")
    return df_pulito

# **Reading & Cleaning Bronze layer**

In [ ]:
print(f"⏳ Lettura da Bronze: {nome_tabella_bronze}")

try:
    df_bronze = spark.read.format("delta").load(f"{bronze_path}/Tables/{nome_tabella_bronze}")
    righe_bronze = df_bronze.count()

    df_silver = cleaning_dataframe(df_bronze)
    righe_silver = df_silver.count()

    print(f"✅ Pulizia completata")
    print(f"├── Righe Bronze : {righe_bronze:,}")
    print(f"├── Righe Silver : {righe_silver:,}")
    print(f"└── Scartate     : {righe_bronze - righe_silver:,}")

except Exception as e:
    print(f"❌ ERRORE in lettura/pulizia di {nome_tabella_bronze}: {e}")
    raise   # it stops the execution


# **Visualize expanded dataframe for attributes**

In [ ]:


# 1. Expand attributes at the 1° level
df_silver_espanso = df_silver.select(
    "*",
    "attributes.*"
).drop("attributes")

# 2. Explode "images" array with one row for each URL
df_silver_finale = df_silver_espanso.select(
    "*",
    F.posexplode("images").alias("image_index", "image_url")
).drop("images")

# 3. Verify schema and visualize preview
print("📐 Schema finale (dopo espansione attributes + esplosione images):")
df_silver_finale.printSchema()

print(f"\n👀 Anteprima dati — {df_silver_finale.count():,} righe totali "
      f"(vs {df_silver.count():,} righe pre-espansione, per via dell'esplosione images)")

display(df_silver_finale.limit(20))


# **Write to Silver layer**

In [ ]:
try:
    righe_scritte = df_silver_finale.count()

    (
        df_silver_finale.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(f"{silver_path}/Tables/{nome_tabella_silver}")
    )

    print(f"✅ Scrittura completata: {nome_tabella_silver}")
    print(f"└── Righe scritte: {righe_scritte:,}")

except Exception as e:
    print(f"❌ ERRORE in scrittura su {nome_tabella_silver}: {e}")
    raise
